# 04 — Récurrences et échéanciers

Toute règle datée que produit cette bibliothèque est **deux décisions indépendantes** :

1. **comment découper** la fenêtre en périodes — l'argument `every` ;
2. **quoi prendre** dans chaque période — l'argument `on`.

`bcal.schedule(...)` est ces deux décisions et rien d'autre. Les huit fonctions nommées
(`last_weekday`, `month_ends`, `imm_dates`…) sont des écritures d'une ligne par-dessus, et
elles restent parce qu'elles disent ce qu'elles font à quelqu'un qui n'a jamais vu la
grammaire.

In [1]:
from datetime import date

import pandas as pd

import better_calendar as bcal
from better_calendar import FRI, MON, SAT, SUN, THU, TUE, WED, Nth, Weekday, periods, schedule


def montre(index):
    return list(index.strftime("%Y-%m-%d"))

## 1. Le moteur

In [2]:
pd.DataFrame(
    [
        {"appel": 'schedule(a, b, "M", "last FRI")',
         "résultat": montre(schedule("2026-01-01", "2026-03-31", "M", "last FRI"))},
        {"appel": 'schedule(a, b, "Q", "2 THU")',
         "résultat": montre(schedule("2026-01-01", "2026-12-31", "Q", "2 THU"))},
        {"appel": 'schedule(a, b, "M", "last B", cal="XNYS")',
         "résultat": montre(schedule("2026-01-01", "2026-03-31", "M", "last B", cal="XNYS"))},
        {"appel": 'schedule(a, b, "M", "3 WED", months=(3,6,9,12))',
         "résultat": montre(
             schedule("2026-01-01", "2026-12-31", "M", "3 WED", months=(3, 6, 9, 12))
         )},
    ]
).set_index("appel")

,résultat
appel,
"schedule(a, b, ""M"", ""last FRI"")","[2026-01-30, 2026-02-27, 2026-03-27]"
"schedule(a, b, ""Q"", ""2 THU"")","[2026-01-08, 2026-04-09, 2026-07-09, 2026-10-08]"
"schedule(a, b, ""M"", ""last B"", cal=""XNYS"")","[2026-01-30, 2026-02-27, 2026-03-31]"
"schedule(a, b, ""M"", ""3 WED"", months=(3,6,9,12))","[2026-03-18, 2026-06-17, 2026-09-16, 2026-12-16]"


### La grammaire `every` — le découpage

Une **unité nue** s'aligne sur le calendrier ; un **multiple** s'ancre sur `start`. C'est
toute la différence entre `"Q"` et `"3M"`, qui durent pourtant trois mois tous les deux.

In [3]:
# Même fenêtre, même durée de période, deux découpages différents.
print('"Q"  trimestre calendaire      :',
      montre(schedule("2026-02-01", "2026-12-31", "Q", "1")))
print('"3M" ancré sur le 1er février  :',
      montre(schedule("2026-02-01", "2026-12-31", "3M", "1")))

"Q"  trimestre calendaire      : ['2026-04-01', '2026-07-01', '2026-10-01']
"3M" ancré sur le 1er février  : ['2026-02-01', '2026-05-01', '2026-08-01', '2026-11-01']


Il n'y a donc **pas** de paramètre `anchor` : le choix est déjà dans la chaîne de
fréquence. `"D"`, `"W"`, `"M"`, `"Q"`, `"Y"` et leurs multiples couvrent tout.

### La grammaire `on` — la sélection

In [4]:
fenetre = ("2026-01-01", "2026-03-31")
pd.DataFrame(
    [
        {"on": o, "sélection": montre(schedule(*fenetre, "M", o))}
        for o in ("1", "15", "last", "-2", "1 B", "last B",
                  "1st FRI", "2 THU", "last FRI", "-2 WED")
    ]
).set_index("on")

,sélection
on,
1,"[2026-01-01, 2026-02-01, 2026-03-01]"
15,"[2026-01-15, 2026-02-15, 2026-03-15]"
last,"[2026-01-31, 2026-02-28, 2026-03-31]"
-2,"[2026-01-30, 2026-02-27, 2026-03-30]"
1 B,"[2026-01-01, 2026-02-02, 2026-03-02]"
last B,"[2026-01-30, 2026-02-27, 2026-03-31]"
1st FRI,"[2026-01-02, 2026-02-06, 2026-03-06]"
2 THU,"[2026-01-08, 2026-02-12, 2026-03-12]"
last FRI,"[2026-01-30, 2026-02-27, 2026-03-27]"


Négatif compte depuis la fin partout. Les suffixes ordinaux sont cosmétiques : `"2 THU"` et
`"2nd THU"` se lisent pareil. Pour du code plutôt que de la configuration, la forme typée
équivalente survit à un renommage :

In [5]:
print(montre(schedule(*fenetre, "M", Nth(-1, FRI))) == montre(schedule(*fenetre, "M", "last FRI")))
print(montre(schedule(*fenetre, "M", Nth(1, "B"))) == montre(schedule(*fenetre, "M", "1 B")))
print("Nth(-1, FRI) se réécrit :", str(Nth(-1, FRI)))

# parse_selector expose la forme analysée, utile pour valider une config avant de l'exécuter.
print("parse_selector('2nd THU') :", bcal.parse_selector("2nd THU"))
print("parse_selector('edges')   :", bcal.parse_selector("edges"), "— la sentinelle bcal.EDGES")
print("mémoïsé                   :", bcal.parse_selector("last") is bcal.parse_selector("last"))

True
True
Nth(-1, FRI) se réécrit : last FRI
parse_selector('2nd THU') : 2 THU
parse_selector('edges')   : Edges() — la sentinelle bcal.EDGES
mémoïsé                   : True


### Ouvré **ou** calendaire : deux axes indépendants

C'est là que les erreurs se logent. `on="last B"` **compte** des jours ouvrés ; `roll=`
**déplace** un résultat sur un jour ouvré. Les deux coïncident souvent, ce qui est
précisément pourquoi la différence doit être écrite plutôt que devinée.

In [6]:
pd.DataFrame(
    [
        {"écriture": 'on="last B"', "sens": "dernier jour OUVRÉ du mois",
         "janvier 2026": montre(
             schedule("2026-01-01", "2026-01-31", "M", "last B", cal="XNYS"))},
        {"écriture": 'on="last", roll="P"', "sens": "dernier jour calendaire, RAMENÉ",
         "janvier 2026": montre(
             schedule("2026-01-01", "2026-01-31", "M", "last", cal="XNYS", roll="P"))},
        {"écriture": 'on="3 B"', "sens": "3e jour OUVRÉ du mois",
         "janvier 2026": montre(
             schedule("2026-01-01", "2026-01-31", "M", "3 B", cal="XNYS"))},
        {"écriture": 'on="3", roll="F"', "sens": "le 3 janvier, poussé en avant",
         "janvier 2026": montre(
             schedule("2026-01-01", "2026-01-31", "M", "3", cal="XNYS", roll="F"))},
    ]
).set_index("écriture")

,sens,janvier 2026
écriture,,
"on=""last B""",dernier jour OUVRÉ du mois,[2026-01-30]
"on=""last"", roll=""P""","dernier jour calendaire, RAMENÉ",[2026-01-30]
"on=""3 B""",3e jour OUVRÉ du mois,[2026-01-06]
"on=""3"", roll=""F""","le 3 janvier, poussé en avant",[2026-01-05]


Les deux premières lignes coïncident ; les deux dernières non. Dès que l'ordinal n'est plus
le dernier, les deux écritures divergent — d'où l'intérêt de dire laquelle on veut.

### Occurrence absente : `missing=`

In [7]:
pd.DataFrame(
    [
        {"missing": "skip (défaut)",
         "nb de mois retenus": len(schedule("2026-01-01", "2026-12-31", "M", "31")),
         "janvier → avril": montre(schedule("2026-01-01", "2026-04-30", "M", "31"))},
        {"missing": "clamp",
         "nb de mois retenus": len(
             schedule("2026-01-01", "2026-12-31", "M", "31", missing="clamp")),
         "janvier → avril": montre(
             schedule("2026-01-01", "2026-04-30", "M", "31", missing="clamp"))},
    ]
).set_index("missing")

,nb de mois retenus,janvier → avril
missing,,
skip (défaut),7,"[2026-01-31, 2026-03-31]"
clamp,12,"[2026-01-31, 2026-02-28, 2026-03-31, 2026-04-30]"


`clamp` rend exprimable « payable le 31 de chaque mois » en un seul appel. `raise` refuse
en nommant la période fautive :

In [8]:
try:
    schedule("2026-01-01", "2026-12-31", "M", "31", missing="raise")
except bcal.ScheduleError as exc:
    print(exc)

The period starting 2026-02-01 has no day 31, and missing='raise' forbids skipping it. Use missing='skip' to drop such periods, or missing='clamp' to take the nearest occurrence that does exist.


### Plusieurs sélecteurs à la fois

In [9]:
print("le 1er et le 15 :", montre(schedule("2026-01-01", "2026-02-28", "M", ["1", "15"])))
print("premier et dernier ouvré :",
      montre(schedule("2026-01-01", "2026-02-28", "M", ["1 B", "last B"], cal="XNYS")))

le 1er et le 15 : ['2026-01-01', '2026-01-15', '2026-02-01', '2026-02-15']
premier et dernier ouvré : ['2026-01-02', '2026-01-30', '2026-02-02', '2026-02-27']


## 2. Les deux exemples fondateurs

Les questions qui ont motivé la bibliothèque, dans leur écriture nommée :

In [10]:
derniers_vendredis = bcal.last_weekday("2026-01-01", "2026-12-31", FRI)
pd.DataFrame(
    {"date": montre(derniers_vendredis), "jour": derniers_vendredis.strftime("%A")}
).set_index("date")

,jour
date,
2026-01-30,Friday
2026-02-27,Friday
2026-03-27,Friday
2026-04-24,Friday
2026-05-29,Friday
2026-06-26,Friday
2026-07-31,Friday
2026-08-28,Friday
2026-09-25,Friday


In [11]:
montre(bcal.nth_weekday("2026-01-01", "2026-12-31", 2, THU, freq="Q"))

['2026-01-08', '2026-04-09', '2026-07-09', '2026-10-08']

Chaque fonction nommée **est** son écriture générique — un test l'épingle, sinon la
documentation mentirait :

In [12]:
fenetre = ("2026-01-01", "2026-12-31")
equivalences = [
    ("last_weekday(a, b, FRI)", bcal.last_weekday(*fenetre, FRI),
     'schedule(a, b, "M", "last FRI")', schedule(*fenetre, "M", "last FRI")),
    ("month_ends(a, b)", bcal.month_ends(*fenetre),
     'schedule(a, b, "M", "last")', schedule(*fenetre, "M", "last")),
    ("month_ends(a, b, cal=…)", bcal.month_ends(*fenetre, cal="XNYS"),
     'schedule(a, b, "M", "last B", cal=…)', schedule(*fenetre, "M", "last B", cal="XNYS")),
    ("quarter_ends(a, b)", bcal.quarter_ends(*fenetre),
     'schedule(a, b, "Q", "last")', schedule(*fenetre, "Q", "last")),
    ("imm_dates(a, b)", bcal.imm_dates(*fenetre),
     'schedule(a, b, "M", "3 WED", months=…)',
     schedule(*fenetre, "M", "3 WED", months=(3, 6, 9, 12))),
]
pd.DataFrame(
    [{"nommé": n, "générique": g, "identique": montre(a) == montre(b)}
     for n, a, g, b in equivalences]
).set_index("nommé")

,générique,identique
nommé,,
"last_weekday(a, b, FRI)","schedule(a, b, ""M"", ""last FRI"")",True
"month_ends(a, b)","schedule(a, b, ""M"", ""last"")",True
"month_ends(a, b, cal=…)","schedule(a, b, ""M"", ""last B"", cal=…)",True
"quarter_ends(a, b)","schedule(a, b, ""Q"", ""last"")",True
"imm_dates(a, b)","schedule(a, b, ""M"", ""3 WED"", months=…)",True


## 3. Trois conventions à connaître

### `n` est 1-based, et négatif compte depuis la fin

C'est tout l'intérêt : « le dernier vendredi du mois » est ce qu'on demande réellement, et
le calculer à la main est précisément là où sont les bugs.

In [13]:
pd.DataFrame(
    [{"n": n, "résultat": montre(bcal.nth_weekday("2026-01-01", "2026-01-31", n, FRI))}
     for n in (1, 2, 3, 4, 5, -1, -2, -5)]
).set_index("n")

,résultat
n,
1,[2026-01-02]
2,[2026-01-09]
3,[2026-01-16]
4,[2026-01-23]
5,[2026-01-30]
-1,[2026-01-30]
-2,[2026-01-23]
-5,[2026-01-02]


### Une occurrence absente est sautée silencieusement

Février a rarement un cinquième vendredi. Lever ferait de l'API un outil inutilisable sur
n'importe quelle plage réelle — d'où `missing="skip"` par défaut, et `"clamp"` quand on
veut l'occurrence la plus proche à la place.

In [14]:
cinquiemes = bcal.nth_weekday("2026-01-01", "2026-12-31", 5, FRI)
print(f"{len(cinquiemes)} mois sur 12 ont un cinquième vendredi :")
print(montre(cinquiemes))
print("\navec clamp, février retombe sur le quatrième :",
      montre(schedule("2026-02-01", "2026-02-28", "M", "5 FRI", missing="clamp")))

4 mois sur 12 ont un cinquième vendredi :
['2026-01-30', '2026-05-29', '2026-07-31', '2026-10-30']

avec clamp, février retombe sur le quatrième : ['2026-02-27']


### L'occurrence appartient à la période, pas à la fenêtre

« Le dernier vendredi de janvier » est une propriété de janvier. Interroger à partir du 15
renvoie quand même le 30 ; une occurrence qui tombe avant la fenêtre est filtrée.

In [15]:
print("depuis le 15 janvier, dernier vendredi :",
      montre(bcal.last_weekday("2026-01-15", "2026-01-31", FRI)))
print("depuis le 15 janvier, premier vendredi :",
      montre(bcal.nth_weekday("2026-01-15", "2026-01-31", 1, FRI)))

depuis le 15 janvier, dernier vendredi : ['2026-01-30']
depuis le 15 janvier, premier vendredi : []


## 4. Les constantes de jour

Un `IntEnum` aligné sur `date.weekday()` — lundi vaut 0, donc elles s'utilisent partout où
la bibliothèque standard attend un indice de jour.

In [16]:
print("valeurs :", {j.name: int(j) for j in Weekday})
print("date.weekday() du 31 juillet 2026 :", date(2026, 7, 31).weekday(), "==", int(FRI))
print("un week-end :", int(SAT), int(SUN))
print("2e mardi de chaque trimestre :",
      montre(bcal.nth_weekday("2026-01-01", "2026-12-31", 2, TUE, freq="Q")))
print("derniers samedis du trimestre :",
      montre(bcal.last_weekday("2026-01-01", "2026-06-30", SAT, freq="Q")))

valeurs : {'MON': 0, 'TUE': 1, 'WED': 2, 'THU': 3, 'FRI': 4, 'SAT': 5, 'SUN': 6}
date.weekday() du 31 juillet 2026 : 4 == 4
un week-end : 5 6
2e mardi de chaque trimestre : ['2026-01-13', '2026-04-14', '2026-07-14', '2026-10-13']
derniers samedis du trimestre : ['2026-03-28', '2026-06-27']


## 5. Les récurrences qui ont un nom

In [17]:
print("1er jour du mois     :", montre(bcal.nth_day("2026-01-01", "2026-03-31", 1)))
print("dernier jour du mois :", montre(bcal.nth_day("2026-01-01", "2026-03-31", -1)))
print("1er jour ouvré NYSE  :",
      montre(bcal.nth_business_day("2026-01-01", "2026-03-31", 1, cal="XNYS")))
print("dernier ouvré NYSE   :",
      montre(bcal.nth_business_day("2026-01-01", "2026-03-31", -1, cal="XNYS")))

1er jour du mois     : ['2026-01-01', '2026-02-01', '2026-03-01']
dernier jour du mois : ['2026-01-31', '2026-02-28', '2026-03-31']
1er jour ouvré NYSE  : ['2026-01-02', '2026-02-02', '2026-03-02']
dernier ouvré NYSE   : ['2026-01-30', '2026-02-27', '2026-03-31']


`month_ends` illustre le point de conception : passer `cal` change la **question**, pas
seulement la réponse. Le 31 janvier et le 28 février 2026 sont des samedis.

In [18]:
pd.DataFrame(
    {
        "calendaire": montre(bcal.month_ends("2026-01-01", "2026-06-30")),
        "ouvré (NYSE)": montre(bcal.month_ends("2026-01-01", "2026-06-30", cal="XNYS")),
        "ouvré (TARGET2)": montre(
            bcal.month_ends("2026-01-01", "2026-06-30", cal="fin:TARGET2")),
    }
)

,calendaire,ouvré (NYSE),ouvré (TARGET2)
0,2026-01-31,2026-01-30,2026-01-30
1,2026-02-28,2026-02-27,2026-02-27
2,2026-03-31,2026-03-31,2026-03-31
3,2026-04-30,2026-04-30,2026-04-30
4,2026-05-31,2026-05-29,2026-05-29
5,2026-06-30,2026-06-30,2026-06-30


In [19]:
print("trimestres          :", montre(bcal.quarter_ends("2026-01-01", "2026-12-31")))
print("exercice fiscal fév :",
      montre(bcal.quarter_ends("2026-01-01", "2026-12-31", anchor_month=2)))
print("années              :", montre(bcal.year_ends("2025-01-01", "2027-06-30")))

trimestres          : ['2026-03-31', '2026-06-30', '2026-09-30', '2026-12-31']
exercice fiscal fév : ['2026-02-28', '2026-05-31', '2026-08-31', '2026-11-30']
années              : ['2025-12-31', '2026-12-31']


### IMM et expirations d'options

Les dates IMM sont le 3e mercredi de mars / juin / septembre / décembre — c'est-à-dire du
**dernier mois du trimestre**, pas du trimestre. La nuance compte, et c'est exactement ce
que le filtre `months=` exprime :

In [20]:
print("3e mercredi du trimestre :",
      montre(bcal.nth_weekday("2026-01-01", "2026-03-31", 3, WED, freq="Q")))
print("IMM (3e mercredi du mois):", montre(bcal.imm_dates("2026-01-01", "2026-03-31")))
print()
print("IMM 2026-2027 :", montre(bcal.imm_dates("2026-01-01", "2027-12-31")))

3e mercredi du trimestre : ['2026-01-21']
IMM (3e mercredi du mois): ['2026-03-18']

IMM 2026-2027 : ['2026-03-18', '2026-06-17', '2026-09-16', '2026-12-16', '2027-03-17', '2027-06-16', '2027-09-15', '2027-12-15']


Les expirations d'options sont le 3e vendredi, ajusté **en arrière** — quand le 3e vendredi
est le Vendredi saint, l'expiration recule au jeudi. C'était le cas en avril 2022 :

In [21]:
pd.DataFrame(
    {
        "3e vendredi brut": montre(bcal.nth_weekday("2022-01-01", "2022-06-30", 3, FRI)),
        "expiration NYSE": montre(bcal.option_expiries("2022-01-01", "2022-06-30", cal="XNYS")),
    }
)

,3e vendredi brut,expiration NYSE
0,2022-01-21,2022-01-21
1,2022-02-18,2022-02-18
2,2022-03-18,2022-03-18
3,2022-04-15,2022-04-14
4,2022-05-20,2022-05-20
5,2022-06-17,2022-06-17


## 6. `on="edges"` — les échéanciers de coupons

Au lieu d'un jour **dans** chaque période, `"edges"` renvoie les **bornes** des périodes :
une date de plus qu'il n'y a de périodes. C'est ce qui transforme `schedule` en générateur
d'échéancier.

Et la règle qui structure tout le module : **le résultat ne dépend d'aucun calendrier tant
que `roll` n'est pas passé.**

In [22]:
brut = schedule("2026-02-28", "2027-08-31", "6M", "edges", eom=True)
ajuste = schedule("2026-02-28", "2027-08-31", "6M", "edges", eom=True, cal="XNYS", roll="MF")

pd.DataFrame({"contractuel (sans roll)": montre(brut), "paiement (roll=MF)": montre(ajuste)})

,contractuel (sans roll),paiement (roll=MF)
0,2026-02-28,2026-02-27
1,2026-08-31,2026-08-31
2,2027-02-28,2027-02-26
3,2027-08-31,2027-08-31


La séparation est la raison d'être du design : un système en aval qui détient un trade
bookié l'an dernier doit pouvoir vérifier que son coupon du 15 mars est bien la même date
contractuelle que la nôtre, même si un férié a déplacé la date de paiement effective. Si
les dates contractuelles dépendaient des données de fériés, régénérer un snapshot ferait
apparaître le **contrat** comme modifié.

Démontrons-le : sans `roll`, le résultat est identique à travers quatre calendriers, même
avec un férié posé pile sur un coupon.

In [23]:
from better_calendar import Calendar

calendriers = {
    "aucun (weekday)": None,
    "XNYS": "XNYS",
    "TARGET2": "fin:TARGET2",
    "férié sur le coupon": Calendar("odd", holidays=["2026-08-31", "2026-09-01"]),
}

pd.DataFrame(
    {
        etiquette: {
            "sans roll": montre(
                schedule("2026-02-28", "2027-08-31", "6M", "edges", eom=True, cal=cal)),
            "avec roll=MF": montre(
                schedule("2026-02-28", "2027-08-31", "6M", "edges", eom=True,
                         cal=cal, roll="MF")),
        }
        for etiquette, cal in calendriers.items()
    }
).T

,sans roll,avec roll=MF
aucun (weekday),"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-31, 2027-02-26, 2027-08-31]"
XNYS,"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-31, 2027-02-26, 2027-08-31]"
TARGET2,"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-31, 2027-02-26, 2027-08-31]"
férié sur le coupon,"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-28, 2027-02-26, 2027-08-31]"


### Les stubs

Un échéancier tombe rarement juste. Le *stub* est ce qu'on fait du reste. Un terme de cinq
mois à fréquence trimestrielle laisse deux mois à placer :

In [24]:
pd.DataFrame(
    [
        {"stub": stub,
         "dates": montre(schedule("2026-01-15", "2026-06-15", "3M", "edges", stub=stub))}
        for stub in ("short_front", "long_front", "short_back", "long_back")
    ]
).set_index("stub")

,dates
stub,
short_front,"[2026-01-15, 2026-03-15, 2026-06-15]"
long_front,"[2026-01-15, 2026-06-15]"
short_back,"[2026-01-15, 2026-04-15, 2026-06-15]"
long_back,"[2026-01-15, 2026-06-15]"


Le choix du stub détermine **de quel bout** la grille régulière est mesurée : un stub
*front* ancre sur la date de fin et remonte, un stub *back* ancre sur le début et descend.
C'est ce qui fait que les coupons tombent sur la maturité au lieu de dériver.

In [25]:
print("front (ancré sur la fin)  :",
      montre(schedule("2026-01-10", "2027-01-15", "6M", "edges", stub="short_front")))
print("back  (ancré sur le début):",
      montre(schedule("2026-01-10", "2027-01-15", "6M", "edges", stub="short_back")))

front (ancré sur la fin)  : ['2026-01-10', '2026-01-15', '2026-07-15', '2027-01-15']
back  (ancré sur le début): ['2026-01-10', '2026-07-10', '2027-01-10', '2027-01-15']


In [26]:
# `none` refuse un terme qui ne tombe pas juste, au lieu d'inventer un stub.
try:
    schedule("2026-01-15", "2026-06-15", "3M", "edges", stub="none")
except bcal.ScheduleError as exc:
    print(exc)

2026-01-15 to 2026-06-15 is not a whole number of 3M periods, and stub='none' forbids a stub. Choose a stub convention, or move one of the dates.


### Les dates sont mesurées depuis l'ancre, jamais pas à pas

Sinon le 31 janvier glisserait au 28 février puis **resterait** sur le 28 pour la vie du
trade. Ici, il revient au 31 dès que le mois le permet :

In [27]:
montre(schedule("2026-01-31", "2026-06-30", "1M", "edges", stub="short_back"))

['2026-01-31',
 '2026-02-28',
 '2026-03-31',
 '2026-04-30',
 '2026-05-31',
 '2026-06-30']

### Périodes d'accrual

`periods()` renvoie les intervalles entre dates consécutives, semi-ouverts `[début, fin)`
pour que les périodes pavent sans compter deux fois la date de bascule.

In [28]:
pd.DataFrame(
    [
        {"début": p.start, "fin": p.end, "jours calendaires": len(p),
         "jours ouvrés": len(p.business_days("XNYS"))}
        for p in periods("2026-01-15", "2027-01-15", "3M", cal="XNYS", roll="MF")
    ]
)

,début,fin,jours calendaires,jours ouvrés
0,2026-01-15,2026-04-15,90,61
1,2026-04-15,2026-07-15,91,62
2,2026-07-15,2026-10-15,92,65
3,2026-10-15,2027-01-15,92,63


`periods()` accepte n'importe quel sélecteur, pas seulement `"edges"` :

In [29]:
mensuel = periods("2026-01-01", "2026-06-30", "M", "last")
print(f"{len(mensuel)} intervalles de fin de mois à fin de mois")
print([(str(p.start), str(p.end)) for p in mensuel[:3]])

5 intervalles de fin de mois à fin de mois
[('2026-01-31', '2026-02-28'), ('2026-02-28', '2026-03-31'), ('2026-03-31', '2026-04-30')]


## 7. Un cas complet : coupons d'une obligation

Semestriel, du 15 mars 2026 au 15 mars 2031, réglé en zone euro.

In [30]:
contractuel = schedule("2026-03-15", "2031-03-15", "6M", "edges")
paiement = schedule("2026-03-15", "2031-03-15", "6M", "edges",
                    cal="fin:TARGET2", roll="MF")

coupons = pd.DataFrame({"contractuel": montre(contractuel), "paiement": montre(paiement)})
coupons["décalé ?"] = coupons["contractuel"] != coupons["paiement"]
coupons["jours ouvrés depuis le précédent"] = [None] + [
    bcal.count(a, b, cal="fin:TARGET2")
    for a, b in zip(coupons["paiement"], coupons["paiement"][1:])
]
coupons

,contractuel,paiement,décalé ?,jours ouvrés depuis le précédent
0,2026-03-15,2026-03-16,True,NaN
1,2026-09-15,2026-09-15,False,128.0
2,2027-03-15,2027-03-15,False,127.0
3,2027-09-15,2027-09-15,False,130.0
4,2028-03-15,2028-03-15,False,130.0
5,2028-09-15,2028-09-15,False,129.0
6,2029-03-15,2029-03-15,False,126.0
7,2029-09-15,2029-09-17,True,129.0
8,2030-03-15,2030-03-15,False,126.0
9,2030-09-15,2030-09-16,True,128.0


## Récapitulatif

| Appel | Rôle |
|---|---|
| `schedule(a, b, every, on, …)` | le moteur : découper, puis choisir |
| `every` | `"D"` `"W"` `"M"` `"Q"` `"Y"` ou un multiple (`"3M"`) |
| `on` | `"last"`, `"15"`, `"1 B"`, `"2 THU"`, `"edges"`, ou une liste |
| `missing=` | `"skip"` (défaut), `"clamp"`, `"raise"` |
| `months=` | restreindre à certains mois calendaires (IMM) |
| `stub=`, `eom=` | seulement avec `on="edges"` |
| `periods(...)` | les intervalles entre dates consécutives |
| `nth_weekday`, `last_weekday`, `nth_day`, `nth_business_day` | raccourcis nommés |
| `month_ends`, `quarter_ends`, `year_ends`, `imm_dates`, `option_expiries` | idem |
| `MON`…`SUN`, `Weekday`, `Nth` | constantes et forme typée du sélecteur |

**Suite :** [05 — Fuseaux et sessions](05-fuseaux-et-sessions.ipynb)